# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 0. Setup — connect to the warehouse (Colab or local)

Skills loaded for this task: `writing-data-contracts` + `flyrank/flyrank-data`.

**Lane 2 — Refresh / Content Opportunity Scoring** (same lane as W01/W02). This notebook moves
off the 30k-row starter CSV onto the real warehouse: `fact_content_daily_performance` for
month=2026-03 (the assignment's specified mid-panel month), joined to `dim_content`.

Token rule: never paste an HF token into a cell (this repo is public). Colab reads it from the
Secrets panel (`HF_TOKEN`); locally it's read from the `HF_TOKEN` environment variable; if
neither is set you'll be prompted with `getpass` so it never lands in the notebook file itself.


In [1]:
import os
import sys
import getpass

import duckdb
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("HF_TOKEN (read token, gated-repo access): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/data_0.parquet')"
DIM_CONTENT = f"read_parquet('{WAREHOUSE}/dim_content.parquet')"

# Cheap sanity ping — metadata only, near-free even on a gated remote table.
ping = con.sql(f"SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date FROM {FACT_MARCH}").df()
print(ping.to_string(index=False))


 n_rows   min_date   max_date
9841378 2026-03-01 2026-03-31


## 1. Unit of analysis + time window

*One row = one WHAT, over which dates? State it, then verify it below.*

**Two grains are in play here, on purpose:**

- **Raw table grain:** one row of `fact_content_daily_performance` = one content page's search/analytics
  performance **on one calendar day**, keyed by `(report_date, client_hash_id, content_hash_id)`.
  Month partition `month=2026-03` (2026-03-01 → 2026-03-31), the assignment's mid-panel month.
- **My lane's feature-frame grain (built from the raw grain):** one row = one content page,
  **scored at the decision point 2026-03-16**, by aggregating its own March 1–15 daily rows
  (the "past" the reviewer would actually have on that date) into a page-level summary. This is
  the same unit W01/W02 committed to — one content page (`content_hash_id`) inside one client
  (`client_hash_id`, grouping only).


In [2]:
# Grain probe on the RAW table: (report_date, client_hash_id, content_hash_id) should be unique.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MARCH}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()

print("Duplicate (report_date, client_hash_id, content_hash_id) groups found:", len(grain_check))
print("Grain holds (one row really is one page-day) — expect 0 rows above." if len(grain_check) == 0 else "GRAIN BROKEN — investigate.")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (report_date, client_hash_id, content_hash_id) groups found: 0
Grain holds (one row really is one page-day) — expect 0 rows above.


,report_date,client_hash_id,content_hash_id,c


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**What I'd predict or rank (label / proxy):** `is_declining` — 1 if a page's search impressions in
the post-decision window (March 17–31) are lower than in the pre-decision window (March 1–15),
else 0. This is the warehouse-native version of the starter CSV's `trend_direction == "down"`
proxy, except built honestly as **prior window → future window** instead of one same-window rule
— the stronger design W02 flagged as the capstone direction.

| Field | Bucket | Why |
|---|---|---|
| `impressions_first_half`, `clicks_first_half`, `ctr_first_half`, `avg_position_first_half`, `content_age_days_at_decision` | **Feature** | Built only from March 1–15 (or from content metadata that predates March 16) — see the "available when?" line on each in §3. |
| `is_declining` | **Label / proxy** | The thing I predict. Never a feature. |
| `impressions_second_half`, `impression_change_pct` | **Label-derived — never a feature** | `impressions_second_half` is the raw post-decision quantity the label is compared against; `impression_change_pct` is the exact percentage the label thresholds on. Both are the answer in disguise — this is the trap I deliberately trigger and remove in §3. |
| `client_hash_id`, `content_hash_id`, `report_date` | **Context** | Grouping, joining, and windowing only — pseudonymous IDs and calendar dates carry no signal the model should learn from. |
| `ga4_*` columns (pageviews, sessions, engaged_sessions, ai_sessions, …) | **Excluded** | `ga4_data_available IS TRUE` for well under half of March rows (verified in §3) — too sparse this round to trust as a feature; a blind fillna(0) would silently encode "GA4 not connected yet" as "zero engagement." Revisit once availability is filtered per-client. |
| `gsc_sum_position` | **Excluded** | Raw numerator behind `gsc_avg_position` — redundant with the ratio I already use, adds nothing. |
| `sessions_organic/direct/referral/social/paid`, `ai_chatgpt/perplexity/gemini/copilot/claude/meta/other` | **Excluded** | Channel/AI-referrer breakdowns are GA4-sourced (same sparsity issue) and out of scope for this lane's five-feature slice — noted, not used. |


## 3. Verify it with queries (grain, counts, missing values, windows) — then five features and the trap

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query A (grain)** ran in §1 already and came back empty — the raw table's grain holds.

**Query B — slice row count and date span** for month=2026-03.


In [3]:
counts = con.sql(f"""
    SELECT
        COUNT(*)                        AS n_rows,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_pages,
        MIN(report_date)                AS min_date,
        MAX(report_date)                AS max_date
    FROM {FACT_MARCH}
""").df()
print(counts.to_string(index=False))


 n_rows  n_clients  n_pages   min_date   max_date
9841378         55   331437 2026-03-01 2026-03-31


**Query C — availability**, filtered with `IS TRUE` (per the flyrank-data skill: GSC/GA4 zeros
before a client's data-start date are zero-*filled*, not "no engagement" — the flag is what's
trustworthy, not the raw number).

In [4]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS both_available_rows
    FROM {FACT_MARCH}
""").df()
print(availability.to_string(index=False))
gsc_rate = availability["gsc_available_rows"][0] / availability["total_rows"][0]
ga4_rate = availability["ga4_available_rows"][0] / availability["total_rows"][0]
print(f"\nGSC available: {gsc_rate:.1%} of March page-day rows | GA4 available: {ga4_rate:.1%} of March page-day rows")
print("This is why GA4 columns are Excluded above, not Features: too many rows are structurally unavailable this month.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  gsc_available_rows  ga4_available_rows  both_available_rows
    9841378           3611061.0            413966.0             364347.0

GSC available: 36.7% of March page-day rows | GA4 available: 4.2% of March page-day rows
This is why GA4 columns are Excluded above, not Features: too many rows are structurally unavailable this month.


### Five features (max) — the lane-2 page-level frame for March 2026

Decision point: **2026-03-16**. Features come only from **March 1–15** (or from content metadata
that predates the decision). March 16 is left as a one-day buffer; the label comes from
**March 17–31**, a window the reviewer cannot see yet on decision day.


In [5]:
feature_query = f"""
WITH page_agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15')                       AS impressions_first_half,
        SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '2026-03-15')                       AS clicks_first_half,
        AVG(gsc_avg_position) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_avg_position > 0) AS avg_position_first_half,
        SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-17')                       AS impressions_second_half
    FROM {FACT_MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
)
SELECT
    p.*,
    DATE_DIFF('day', c.content_created_date, DATE '2026-03-16') AS content_age_days_at_decision
FROM page_agg p
LEFT JOIN {DIM_CONTENT} c
  ON p.client_hash_id = c.client_hash_id AND p.content_hash_id = c.content_hash_id
WHERE p.impressions_first_half > 0   -- lane 2: 'pages with measurable search demand' (W01 framing)
"""

lane = con.sql(feature_query).df()
print(f"Pages with measurable demand in the pre-decision window: {len(lane):,}")

# Pages with zero available rows in the post-decision window have no observable label — drop them,
# this is missing (never happened), not a zero outcome.
before = len(lane)
lane = lane.dropna(subset=["impressions_second_half"]).reset_index(drop=True)
print(f"Dropped for unobservable post-decision label: {before - len(lane):,} -> {len(lane):,} pages remain")

# avg_position_first_half is NaN when a page had impressions but zero ranked days in the window —
# same convention as the starter CSV: 0 means 'no position data', not position zero.
lane["avg_position_first_half"] = lane["avg_position_first_half"].fillna(0)
lane["ctr_first_half"] = (lane["clicks_first_half"] / lane["impressions_first_half"]).replace([np.inf, -np.inf], 0)

FEATURES = ["impressions_first_half", "clicks_first_half", "ctr_first_half",
            "avg_position_first_half", "content_age_days_at_decision"]
lane[["client_hash_id", "content_hash_id"] + FEATURES].head(5)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages with measurable demand in the pre-decision window: 151,981
Dropped for unobservable post-decision label: 11,120 -> 140,861 pages remain


,client_hash_id,content_hash_id,impressions_first_half,clicks_first_half,ctr_first_half,avg_position_first_half,content_age_days_at_decision
0,client_400c21c81c8b46ef,content_8d8395577b139cf2,14.0,0.0,0.000000,5.190476,229
1,client_400c21c81c8b46ef,content_e1ca51daf16dc7e0,84.0,0.0,0.000000,6.544139,173
2,client_400c21c81c8b46ef,content_036d16b5002a1e66,285.0,0.0,0.000000,4.734732,173
3,client_400c21c81c8b46ef,content_b39b7bc4b9a7db7e,539.0,0.0,0.000000,7.548281,173
4,client_400c21c81c8b46ef,content_1d791a6f3afefbea,285.0,1.0,0.003509,6.488304,173


**Five features, one "knowable at the decision moment because…" line each:**

1. **`impressions_first_half`** — search impressions summed over March 1–15. Knowable because: these
   are already-recorded GSC impressions from before the March 16 decision point; nothing in the sum
   requires seeing what happens afterward.
2. **`clicks_first_half`** — search clicks summed over the same window. Knowable because: same as
   above — purely observed, pre-decision counts.
3. **`ctr_first_half`** — `clicks_first_half / impressions_first_half`. Knowable because: it's a
   ratio of two quantities that are themselves both already knowable pre-decision.
4. **`avg_position_first_half`** — mean GSC position over March 1–15 days that had a ranked
   position (0 when the page never ranked in that window, matching the starter dictionary's
   "0 = no data" convention). Knowable because: observed ranking history up to the decision date.
5. **`content_age_days_at_decision`** — days between `content_created_date` and 2026-03-16.
   Knowable because: publish date is a fixed historical fact recorded at creation time, always
   available before any future decision point — it can't move.


### The trap — add one label-derived column, watch the score jump, then remove it

Label: `is_declining` = 1 if impressions **fell** from the pre-decision window to the
post-decision window. Quick score: **Precision@50** on a depth-2 decision tree (same test the
starter notebook 02 used on the CSV's `trend_pct` → `trend_direction` leak) — same lesson, this
time on real warehouse data I pulled myself.


In [6]:
from sklearn.tree import DecisionTreeClassifier, export_text

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

lane["is_declining"] = (lane["impressions_second_half"] < lane["impressions_first_half"]).astype(int)
print(f"Pages scored: {len(lane):,} | declining rate: {lane['is_declining'].mean():.1%}")

y = lane["is_declining"].values
X_honest = lane[FEATURES].fillna(0)

honest_tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_honest, y)
honest_score = honest_tree.predict_proba(X_honest)[:, 1]
print(f"\nHONEST Precision@50 (five features only): {precision_at_k(honest_score, y, 50):.3f}")
print(export_text(honest_tree, feature_names=FEATURES))


Pages scored: 140,861 | declining rate: 44.1%

HONEST Precision@50 (five features only): 0.420
|--- impressions_first_half <= 2.50
|   |--- impressions_first_half <= 1.50
|   |   |--- class: 0
|   |--- impressions_first_half >  1.50
|   |   |--- class: 0
|--- impressions_first_half >  2.50
|   |--- content_age_days_at_decision <= 19.50
|   |   |--- class: 0
|   |--- content_age_days_at_decision >  19.50
|   |   |--- class: 1



In [7]:
# --- THE TRAP: add ONE column derived straight from the label window ---
# impression_change_pct is the exact % the label thresholds on (< 0% -> declining) — the
# same relationship trend_pct had to trend_direction in the starter CSV (notebook 02).
lane["impression_change_pct"] = (
    (lane["impressions_second_half"] - lane["impressions_first_half"]) / lane["impressions_first_half"] * 100
)

X_leaky = lane[FEATURES + ["impression_change_pct"]].fillna(0)
leaky_tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
leaky_score = leaky_tree.predict_proba(X_leaky)[:, 1]
print(f"LEAKY Precision@50 (+ impression_change_pct): {precision_at_k(leaky_score, y, 50):.3f}  <- jumps toward perfect")
print(export_text(leaky_tree, feature_names=FEATURES + ["impression_change_pct"]))
print(
    "\nThe tree just splits on impression_change_pct near 0% and nails the label — because "
    "the label IS a threshold on that column. This is not a better model, it's the answer "
    "smuggled in as a feature."
)


LEAKY Precision@50 (+ impression_change_pct): 1.000  <- jumps toward perfect
|--- impression_change_pct <= -0.01
|   |--- class: 1
|--- impression_change_pct >  -0.01
|   |--- avg_position_first_half <= 0.05
|   |   |--- class: 0
|   |--- avg_position_first_half >  0.05
|   |   |--- class: 0


The tree just splits on impression_change_pct near 0% and nails the label — because the label IS a threshold on that column. This is not a better model, it's the answer smuggled in as a feature.


In [8]:
# --- delete the leak, keep the honest number ---
lane = lane.drop(columns=["impression_change_pct"])
print("impression_change_pct removed. Honest feature set restored:", FEATURES)
print(f"Honest Precision@50 stands: {precision_at_k(honest_score, y, 50):.3f}")
print(f"(vs the leaky {precision_at_k(leaky_score, y, 50):.3f} — that number never goes in a report.)")


impression_change_pct removed. Honest feature set restored: ['impressions_first_half', 'clicks_first_half', 'ctr_first_half', 'avg_position_first_half', 'content_age_days_at_decision']
Honest Precision@50 stands: 0.420
(vs the leaky 1.000 — that number never goes in a report.)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


In [9]:
# One named limitation, shown with a number: GA4 availability in this exact slice.
limit_check = con.sql(f"""
    SELECT
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS gsc_avail_rate,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS ga4_avail_rate
    FROM {FACT_MARCH}
""").df()
print(limit_check.to_string(index=False))


 gsc_avail_rate  ga4_avail_rate
       0.366926        0.042064


**Named limitation — GA4 coverage is too thin and too structural to use this month.**
Only ~4% of March page-day rows have `ga4_data_available IS TRUE`, versus ~37% for GSC (§3
Query C). That gap is not random noise — it follows each client's `ga4_data_start` in
`dim_clients` (the flyrank-data skill's panel warning), so a page's GA4 numbers are missing
*because its client hasn't onboarded GA4 yet*, not because engagement was zero. Two
consequences for this contract: (1) my five-feature frame is GSC-only this round — GA4
engagement signal (sessions, engaged sessions, AI-referral traffic) is a documented gap, not an
oversight; (2) even the GSC-only frame likely over-represents clients with longer/denser search
history, since pages need `impressions_first_half > 0` to enter the slice at all — thin-history
clients are underrepresented in this March cut by construction.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
